In [4]:
import sys
import os

sys.path.append(os.path.abspath("../src"))

import pandas as pd
import numpy as np
from utils import audit, load_and_split_city, diagnose_missing, impute_time_series

# 1. Load dữ liệu cho Bengaluru
bengaluru = load_and_split_city("Bengaluru")
print("=== BENGALURU (audit gốc) ===")
display(audit(bengaluru).sort_values("pct_missing", ascending=False))

# 2. Load dữ liệu cho Amritsar
amritsar = load_and_split_city("Amritsar")
print("\n=== AMRITSAR (audit gốc) ===")
display(audit(amritsar).sort_values("pct_missing", ascending=False))

=== BENGALURU (audit gốc) ===


,dtype,n_missing,pct_missing,n_unique
Xylene,float64,2009,100.0,0
PM10,float64,360,17.9,1521
Benzene,float64,266,13.2,470
NH3,float64,203,10.1,1351
PM2.5,float64,146,7.3,1585
O3,float64,144,7.2,1620
AQI,float64,99,4.9,202
AQI_Bucket,str,99,4.9,5
Toluene,float64,93,4.6,673
CO,float64,11,0.5,390



=== AMRITSAR (audit gốc) ===


,dtype,n_missing,pct_missing,n_unique
NOx,float64,289,23.7,807
Xylene,float64,213,17.4,600
Toluene,float64,212,17.4,422
Benzene,float64,195,16.0,575
SO2,float64,168,13.8,718
PM2.5,float64,128,10.5,1019
O3,float64,115,9.4,903
AQI,float64,95,7.8,246
AQI_Bucket,str,95,7.8,6
NO,float64,93,7.6,884


In [5]:
# 1. Load dữ liệu cho Bengaluru
bengaluru = load_and_split_city("Bengaluru")
print("=== BENGALURU (audit gốc) ===")
display(audit(bengaluru).sort_values("pct_missing", ascending=False))

# 2. Load dữ liệu cho Amritsar
amritsar = load_and_split_city("Amritsar")
print("\n=== AMRITSAR (audit gốc) ===")
display(audit(amritsar).sort_values("pct_missing", ascending=False))

=== BENGALURU (audit gốc) ===


,dtype,n_missing,pct_missing,n_unique
Xylene,float64,2009,100.0,0
PM10,float64,360,17.9,1521
Benzene,float64,266,13.2,470
NH3,float64,203,10.1,1351
PM2.5,float64,146,7.3,1585
O3,float64,144,7.2,1620
AQI,float64,99,4.9,202
AQI_Bucket,str,99,4.9,5
Toluene,float64,93,4.6,673
CO,float64,11,0.5,390



=== AMRITSAR (audit gốc) ===


,dtype,n_missing,pct_missing,n_unique
NOx,float64,289,23.7,807
Xylene,float64,213,17.4,600
Toluene,float64,212,17.4,422
Benzene,float64,195,16.0,575
SO2,float64,168,13.8,718
PM2.5,float64,128,10.5,1019
O3,float64,115,9.4,903
AQI,float64,95,7.8,246
AQI_Bucket,str,95,7.8,6
NO,float64,93,7.6,884


In [6]:
# Bengaluru: Xylene thiếu 100% (2009/2009) -> drop. (đề xuất, cần nhóm đồng ý)
# Amritsar: không có cột thiếu quá nặng -> giữ nguyên toàn bộ cột
#   (nếu Cell 2 đề xuất DROP cột nào cho Amritsar thì thêm vào đây)
bengaluru_drop = ["Xylene"]
amritsar_drop = []

bengaluru_clean, bengaluru_flat_changed = impute_time_series(
    bengaluru, "Bengaluru", drop_cols=bengaluru_drop
)
print()
amritsar_clean, amritsar_flat_changed = impute_time_series(
    amritsar, "Amritsar", drop_cols=amritsar_drop
)

--- Xử lý Missing Values cho Bengaluru ---
  - NOx: phát hiện 327 ngày flatline (0 liên tục >= 3 ngày) -> NaN
  - Benzene: phát hiện 64 ngày flatline (0 liên tục >= 3 ngày) -> NaN
  - Toluene: phát hiện 15 ngày flatline (0 liên tục >= 3 ngày) -> NaN
  - Đã xoá cột (đề xuất, cần nhóm đồng ý): ['Xylene']
  => Còn 1027 ô missing SAU xử lý (CHỦ Ý giữ NaN ở khối trống dài, không bịa số)

--- Xử lý Missing Values cho Amritsar ---
  - CO: phát hiện 254 ngày flatline (0 liên tục >= 3 ngày) -> NaN
  - Toluene: phát hiện 9 ngày flatline (0 liên tục >= 3 ngày) -> NaN
  - Xylene: phát hiện 9 ngày flatline (0 liên tục >= 3 ngày) -> NaN
  => Còn 1035 ô missing SAU xử lý (CHỦ Ý giữ NaN ở khối trống dài, không bịa số)


In [7]:
# Kiểm chứng: so sánh % missing trước / sau cho từng cột
pollutant_cols = ['PM2.5','PM10','NO','NO2','NOx','NH3','CO','SO2','O3','Benzene','Toluene','Xylene']

def compare_missing(before_df, after_df, cols):
    before = before_df[cols].isna().mean() * 100
    after_cols = [c for c in cols if c in after_df.columns]
    after = after_df[after_cols].isna().mean() * 100
    return pd.DataFrame({'truoc_%': before.round(1), 'sau_%': after.reindex(before.index).round(1)})

print("=== BENGALURU: truoc/sau ===")
display(compare_missing(bengaluru, bengaluru_clean, pollutant_cols + ['AQI']))

print("\n=== AMRITSAR: truoc/sau ===")
display(compare_missing(amritsar, amritsar_clean, pollutant_cols + ['AQI']))

=== BENGALURU: truoc/sau ===


,truoc_%,sau_%
PM2.5,7.3,4.0
PM10,17.9,15.8
NO,0.3,0.0
NO2,0.3,0.0
NOx,0.2,15.5
NH3,10.1,5.1
CO,0.5,0.0
SO2,0.3,0.0
O3,7.2,1.4
Benzene,13.2,4.3



=== AMRITSAR: truoc/sau ===


,truoc_%,sau_%
PM2.5,10.5,4.6
PM10,5.2,0.0
NO,7.6,2.9
NO2,3.6,0.0
NOx,23.7,20.9
NH3,3.4,0.0
CO,6.2,13.8
SO2,13.8,4.2
O3,9.4,2.9
Benzene,16.0,8.2


In [8]:
os.makedirs("../data/interim", exist_ok=True)
bengaluru_clean.to_csv("../data/interim/bengaluru_after_missing.csv", index=False)
amritsar_clean.to_csv("../data/interim/amritsar_after_missing.csv", index=False)
print("Đã lưu data/interim/bengaluru_after_missing.csv và data/interim/amritsar_after_missing.csv")
print("Bengaluru:", bengaluru_clean.shape, " | Amritsar:", amritsar_clean.shape)

Đã lưu data/interim/bengaluru_after_missing.csv và data/interim/amritsar_after_missing.csv
Bengaluru: (2009, 28)  | Amritsar: (1221, 29)


In [9]:
os.makedirs("../reports", exist_ok=True)
log_path = "../reports/cleaning_log.csv"

def build_log(city, before_df, after_df, cols, dropped):
    rows = []
    for c in cols:
        n_before = int(before_df[c].isna().sum())
        if c in dropped or c not in after_df.columns:
            n_after, action = None, "dropped"
        else:
            n_after = int(after_df[c].isna().sum())
            action = "imputed/flatline->NaN" if n_after != n_before else "unchanged"
        rows.append({
            "city": city,
            "column": c,
            "n_missing_before": n_before,
            "pct_before": round(n_before / len(before_df) * 100, 1),
            "n_missing_after": n_after,
            "pct_after": None if n_after is None else round(n_after / len(after_df) * 100, 1),
            "action": action,
        })
    return pd.DataFrame(rows)

new_log = pd.concat([
    build_log("Bengaluru", bengaluru, bengaluru_clean, pollutant_cols + ['AQI'], bengaluru_drop),
    build_log("Amritsar", amritsar, amritsar_clean, pollutant_cols + ['AQI'], amritsar_drop),
], ignore_index=True)

# Nếu file log đã có (của bạn cùng nhóm): giữ nguyên dòng cũ, chỉ thay dòng của 2 thành phố này
if os.path.exists(log_path):
    old_log = pd.read_csv(log_path)
    if "city" in old_log.columns:
        old_log = old_log[~old_log["city"].isin(["Bengaluru", "Amritsar"])]
    new_log = pd.concat([old_log, new_log], ignore_index=True)

new_log.to_csv(log_path, index=False)
print("Đã ghi log vào", log_path)
display(new_log[new_log["city"].isin(["Bengaluru", "Amritsar"])])

Đã ghi log vào ../reports/cleaning_log.csv


,Đã drop NH3,Xylene ở Gurugram (92.9% missing,gần như 1 khối liên tục),city,column,n_missing_before,pct_before,n_missing_after,pct_after,action
2,NaN,NaN,NaN,Bengaluru,PM2.5,146.0,7.3,80.0,4.0,imputed/flatline->NaN
3,NaN,NaN,NaN,Bengaluru,PM10,360.0,17.9,318.0,15.8,imputed/flatline->NaN
4,NaN,NaN,NaN,Bengaluru,NO,6.0,0.3,0.0,0.0,imputed/flatline->NaN
5,NaN,NaN,NaN,Bengaluru,NO2,6.0,0.3,0.0,0.0,imputed/flatline->NaN
6,NaN,NaN,NaN,Bengaluru,NOx,4.0,0.2,312.0,15.5,imputed/flatline->NaN
7,NaN,NaN,NaN,Bengaluru,NH3,203.0,10.1,102.0,5.1,imputed/flatline->NaN
8,NaN,NaN,NaN,Bengaluru,CO,11.0,0.5,0.0,0.0,imputed/flatline->NaN
9,NaN,NaN,NaN,Bengaluru,SO2,6.0,0.3,0.0,0.0,imputed/flatline->NaN
10,NaN,NaN,NaN,Bengaluru,O3,144.0,7.2,28.0,1.4,imputed/flatline->NaN
11,NaN,NaN,NaN,Bengaluru,Benzene,266.0,13.2,87.0,4.3,imputed/flatline->NaN
